# LoReFT

Replicates the emoji-chat demo from **"ReFT: Representation Finetuning for Language Models"** ([arXiv:2404.03592](https://arxiv.org/abs/2404.03592)) on Qwen2.5-1.5B-Instruct, end to end in one notebook:

1. **Training** — a rank-4 LoReFT intervention on the block output of layer 8 is trained with pyreft on ten instruction→emoji examples (supervised at the last prompt position only, following the official demo) and saved to `./weight/`.
2. **Steering** — the trained intervention is applied at the last prompt position in vLLM and makes the model answer in emojis.

Uses the EasySteer v2 steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

## Train the intervention

In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

import torch
import transformers

import easysteer.reft.pyreft as pyreft

device = "cuda"

MODEL = "/home/shenyl/hf/model/Qwen/Qwen2.5-1.5B-Instruct/"  # Qwen/Qwen2.5-1.5B-Instruct

model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map=device
)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL, model_max_length=2048, padding_side="right", use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   5%|▍         | 16/338 [00:00<00:02, 146.14it/s]

...

Loading weights:  38%|███▊      | 128/338 [00:00<00:00, 452.91it/s]

Loading weights:  56%|█████▌    | 188/338 [00:00<00:00, 509.53it/s]

Loading weights:  71%|███████   | 240/338 [00:00<00:00, 510.58it/s]

Loading weights:  86%|████████▋ | 292/338 [00:00<00:00, 466.93it/s]

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 466.89it/s]

In [2]:
# Rank-4 LoReFT intervention on the block output of layer 8.
reft_config = pyreft.ReftConfig(
    representations={
        "layer": 8,
        "component": "block_output",
        "low_rank_dimension": 4,
        "intervention": pyreft.LoreftIntervention(
            embed_dim=model.config.hidden_size, low_rank_dimension=4
        ),
    }
)
reft_model = pyreft.get_reft_model(model, reft_config)
reft_model.set_device(device)
reft_model.print_trainable_parameters()

trainable intervention params: 12,292 || trainable model params: 0
model params: 1,543,714,304 || trainable%: 0.0007962613268627198


In [3]:
import json

prompt_no_input_template = "<|im_start|>user\n%s<|im_end|>\n<|im_start|>assistant\n"

with open("training_examples.json", encoding="utf-8") as f:
    training_examples = json.load(f)

# Supervise only the last prompt position — the position the intervention
# is applied to at inference time.
data_module = pyreft.make_last_position_supervised_data_module(
    tokenizer,
    model,
    [prompt_no_input_template % e["instruction"] for e in training_examples],
    [e["emoji"] for e in training_examples],
)

In [4]:
training_args = transformers.TrainingArguments(
    num_train_epochs=200.0,
    output_dir="./weight",
    per_device_train_batch_size=10,
    learning_rate=4e-3,
    logging_steps=40,
    report_to=[],
    save_strategy="no",
    disable_tqdm=True,  # keep the log to the periodic loss lines
)
trainer = pyreft.ReftTrainerForCausalLM(
    model=reft_model, processing_class=tokenizer, args=training_args, **data_module
)
_ = trainer.train()

reft_model.set_device("cpu")  # move to cpu before saving
reft_model.save(save_directory="./weight", save_to_hf_hub=False)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'pad_token_id': 151645}.


{'loss': '2.176', 'grad_norm': '8.864', 'learning_rate': '0.00322', 'epoch': '40'}


{'loss': '0.8766', 'grad_norm': '17.56', 'learning_rate': '0.00242', 'epoch': '80'}


{'loss': '0.5132', 'grad_norm': '6.992', 'learning_rate': '0.00162', 'epoch': '120'}


{'loss': '0.2954', 'grad_norm': '7.378', 'learning_rate': '0.00082', 'epoch': '160'}


{'loss': '0.1799', 'grad_norm': '3.622', 'learning_rate': '2e-05', 'epoch': '200'}
{'train_runtime': '18.95', 'train_samples_per_second': '105.5', 'train_steps_per_second': '10.55', 'train_loss': '0.8082', 'epoch': '200'}


Directory './weight' already exists.


In [5]:
import gc

# Free the HF training model before booting the vLLM engine. Moving
# the weights to CPU first guarantees the ~3 GiB leaves the GPU even
# if a stray reference keeps the model object alive.
model.to("cpu")
del trainer, reft_model, model
gc.collect()
torch.cuda.empty_cache()

## Steering

In [6]:
from vllm import LLM, SamplingParams
import easysteer.vectors as vec
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    steer_algorithms=["loreft"],
    # Headroom for the training process's residual CUDA context, which
    # shares the GPU with the engine in this single-notebook flow.
    gpu_memory_utilization=0.85,
)

# Declared by name alone, loreft resolves conservatively to the split
# graph tier (payload ranks are only known at request time). This
# notebook's intervention is rank-4 — inside the in-graph rank cap —
# so advanced users can pin the faster in-graph tier explicitly:
#
# llm = LLM(
#     model=MODEL,
#     enable_steer_vector=True,
#     steer_algorithms=["loreft"],
#     steer_graph_mode="in_graph",
#     gpu_memory_utilization=0.85,
# )

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


WARNING 08-05 19:22:22 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.29it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.29it/s]
(EngineCore pid=3736977) 


...

Capturing CUDA graphs (PIECEWISE):  67%|██████▋   | 34/51 [00:02<00:01, 15.96it/s]

Capturing CUDA graphs (PIECEWISE):  75%|███████▍  | 38/51 [00:02<00:00, 16.76it/s]

Capturing CUDA graphs (PIECEWISE):  82%|████████▏ | 42/51 [00:02<00:00, 17.03it/s]

Capturing CUDA graphs (PIECEWISE):  90%|█████████ | 46/51 [00:02<00:00, 17.06it/s]

Capturing CUDA graphs (PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 16.62it/s]


In [7]:
prompts = [
    prompt_no_input_template % "Who are you?",
    prompt_no_input_template % "Who am I?",
]
params = SamplingParams(temperature=0, max_tokens=16, skip_special_tokens=False)

baseline = llm.generate(prompts, params, use_tqdm=False)
print("=====Baseline=====")
for out in baseline:
    print(out.outputs[0].text)

=====Baseline=====
I'm an AI assistant created by Anthropic to be helpful, harmless, and
You are a large language model.


In [8]:
# LoReFT intervenes where it was trained: the last prompt position.
steering = SteeringSpec(vectors=[
    VectorSpec(
        data=vec.from_pyreft("./weight/"),
        algorithm="loreft",
        scale=1.0,
        layers=[8],
        apply=ApplySpec(prompt_positions=[-1]),
    ),
])

steered = llm.generate(prompts, params, steering=steering, use_tqdm=False)
print("=====LoReFT Steered=====")
for out in steered:
    print(out.outputs[0].text)

=====LoReFT Steered=====
🤖💬🌐🧠
👤❓🔍🌟
